In [ ]:
import os
import subprocess
import base64
from kaggle_secrets import UserSecretsClient

# 1. Load Secrets
print("Loading API Keys from Kaggle Secrets...")
user_secrets = UserSecretsClient()
os.environ["GEMINI_API_KEY"] = user_secrets.get_secret("GEMINI_API_KEY")
os.environ["PEXELS_API_KEY"] = user_secrets.get_secret("PEXELS_API_KEY")
os.environ["TELEGRAM_BOT_TOKEN"] = user_secrets.get_secret("TELEGRAM_BOT_TOKEN")
os.environ["TELEGRAM_CHAT_ID"] = user_secrets.get_secret("TELEGRAM_CHAT_ID")
os.environ["AUTO_PUBLISH"] = "true"

google_client_secrets_b64 = user_secrets.get_secret("GOOGLE_CLIENT_SECRETS_B64")
google_token_pickle_b64 = user_secrets.get_secret("GOOGLE_TOKEN_PICKLE_B64")

# 2. Clone Repository
print("Cloning GitHub Repository...")
if not os.path.exists("agent-automation"):
    !git clone https://github.com/manojvenaram/agent-automation.git

# 3. Enter directory and setup credentials
%cd /kaggle/working/agent-automation
!mkdir -p credentials

# Decode and write YouTube credentials
with open("credentials/client_secrets.json", "wb") as f:
    f.write(base64.b64decode(google_client_secrets_b64))
    
with open("credentials/token.pickle", "wb") as f:
    f.write(base64.b64decode(google_token_pickle_b64))

# 4. Install Dependencies
print("Installing Python Packages and System Requirements...")
!apt-get update && apt-get install -y ffmpeg
!pip install -r requirements.txt

# 5. Restore Memory from Kaggle Dataset (If it exists)
if os.path.exists("/kaggle/input/shorts-agent-memory"):
    print("Restoring Agent Memory from Dataset...")
    !cp -r /kaggle/input/shorts-agent-memory/* data/

# 6. Run the Agent!
print("Starting Autonomous Agent...")
!python app.py run-agent

# 7. Zip the new memory for you to download (or update dataset)
print("Archiving memory for persistence...")
!zip -r /kaggle/working/agent_memory_backup.zip data/
